In [4]:
#dependencies
%pip install -q pypdf scikit-learn pandas numpy matplotlib seaborn


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
from pypdf import PdfReader
import re, json
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 90)
pd.set_option("display.width", 200)

print("Core setup complete.")

Core setup complete.


In [6]:
## load the pdf file
NPR_PATH = "N_PR_7150_002D_.pdf"
reader = PdfReader(NPR_PATH)

In [7]:
## extract text from the PDF and count the number of pages and characters
raw = "\n".join((p.extract_text() or "") for p in reader.pages)
print(f"Pages: {len(reader.pages)} Characters: {len(raw):,}")

Pages: 89 Characters: 212,399


In [8]:
print(reader.pages[7].extract_text())

Chapter 1: Introduction
1.1 Overview
1.1.1 This directive imposes requirements on procedures, design considerations, activities, and tasks
used to acquire, develop, maintain, operate, retire, and manage applicable software. This directive is
a designed set of requirements for protecting the ’Agency’s investment in software engineering
products and fulfilling our responsibility to the citizens of the United States (U.S.). 
1.1.2 The requirements in this directive have been extracted from industry standards and proven
NASA experience in software engineering. Centers and software developers may show that many of
the requirements are satisfied through existing programs, procedures, and processes. 
1.1.3 The Agency makes significant investments in software engineering to support the Agency’s
investment areas: Space Flight, Aeronautics, Research and Technology, Information Technology
(IT), and Institutional Infrastructure. NASA ensures that programs, projects, systems, and
subsystems that us

In [9]:
##look for the page header beginning with 'NPR 7150...' and ending with 'Page X of 89' and remove it from the text
text = re.sub(r"NPR 7150\.2D --.*?Page \d+ of 89", " ", raw, flags=re.S)

In [10]:
##find the footer "This document does not bind the public..." and remove it from the text
text = re.sub(r"This document does not bind the public.*?nodis3\.gsfc\.nasa\.gov\.", " ", text, flags=re.S)

In [11]:
## replace multiple whitespace characters with a single space and print the number of characters
text = re.sub(r"\s+", " ", text)
print(f"After cleaning: {len(text):,} characters")

After cleaning: 183,947 characters


In [12]:
#Backward-window parse: for each [SWE-###] tag, walk back to the nearest section heading.
SECTION = re.compile(r"(?:^|\s)(\d+(?:\.\d+)+)\s+(?=[A-Za-z\u201c\"(])")
SWE_TAG = re.compile(r"\[SWE-(\d{3})\]")

clauses, seen, cursor = [], set(), 0
for m in SWE_TAG.finditer(text):
    swe = m.group(1)
    if swe in seen:
        continue
    seen.add(swe)
    window = text[cursor:m.start()]
    hits = list(SECTION.finditer(window))
    if hits:
        section, body = hits[-1].group(1), window[hits[-1].end():]
    else:
        section, body = None, window[-600:]
    clauses.append({"swe_id": f"SWE-{swe}", "section": section, "text": body.strip()})
    cursor = m.end()

In [13]:
df_clauses = pd.DataFrame(clauses)

In [21]:
##display the first 10 rows of the dataframe
df_clauses.head(10)

,swe_id,section,text
0,SWE-002,2.1.1.1,The NASA OCE shall lead and maintain a NASA So...
1,SWE-004,2.1.1.2,The NASA OCE shall periodically benchmark each...
2,SWE-152,2.1.1.3,The NASA OCE shall periodically review the pro...
3,SWE-129,2.1.1.4,The NASA OCE shall authorize appraisals agains...
4,SWE-100,2.1.1.5,The NASA OCE and Center training organizations...
5,SWE-098,2.1.1.6,The NASA OCE shall maintain an Agency-wide pro...
6,SWE-208,2.1.2.2,"The NASA Chief, SMA shall lead and maintain a ..."
7,SWE-209,2.1.2.3,"The NASA Chief, SMA shall periodically benchma..."
8,SWE-212,2.1.2.4,"The NASA Chief, SMA shall periodically review ..."
9,SWE-221,2.1.2.5,"The NASA Chief, SMA shall authorize appraisals..."
